# EDA

## Importações

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

import sys
sys.path.append(f"../")
from Airflow_ETL.Pipelines.src.transacoes.criacao_clientes import gerar_clientes_pf, gerar_clientes_pj
from Airflow_ETL.Pipelines.src.transacoes.criacao_transacoes import simular_mes

In [2]:
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 50)

## Funções Auxiliares

In [3]:
# Visualizar distribuições dos dados (histograma)
def visualizar_distribuicoes(df : pd.DataFrame):
    cols = df.columns    
    n_cols = 4
    if len(df.columns)%n_cols > 0:
        n_rows = len(df.columns)//n_cols + 1
    else:
        n_rows = len(df.columns)//n_cols
    fig, axs = plt.subplots(n_rows, n_cols, figsize=(n_cols*3, n_rows*3))
    axs = axs.flatten()

    for i in range(len(df.columns)):
        axs[i].hist(df[cols[i]], bins=100)
        axs[i].set_title(cols[i])

    plt.tight_layout()
    plt.show()
    
# Visualizar métricas R e P-Value entre as variáveis
def visualizar_correlacoes(df):
    df_copy = df.copy()

    # separar colunas
    num_cols = df_copy.select_dtypes(include=[np.number]).columns
    cat_cols = df_copy.select_dtypes(include=['object', 'category']).columns

    # criar dummies das categóricas
    if len(cat_cols) > 0:
        df_cat = pd.get_dummies(df_copy[cat_cols], prefix=cat_cols)
        df_vis = pd.concat([df_copy[num_cols], df_cat], axis=1)
    else:
        df_vis = df_copy[num_cols]

    cols = df_vis.columns
    n = len(cols)

    fig, axs = plt.subplots(n, n, figsize=(n*1.8, n*1.8))

    for i in range(n):
        for j in range(n):
            ax = axs[i, j]

            x = df_vis[cols[j]]
            y = df_vis[cols[i]]

            if i != j:
                ax.scatter(x, y, s=5)
                corr, p_value = stats.spearmanr(x, y)
                ax.set_title(f"r={corr:.2f} | p={p_value:.2g}", fontsize=7)
            else:
                ax.text(0.5, 0.5, cols[i], ha='center', va='center')

            ax.set_xticks([])
            ax.set_yticks([])

    plt.tight_layout()
    plt.show()

# Estudo

In [4]:
clientes_pj = gerar_clientes_pj(1000)
clientes_pj.describe()

,idade_empresa,faturamento_anual,num_funcionarios,margem_lucro,alavancagem,historico_atraso,pd
count,1000.000000,1.000000e+03,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000
mean,9.717000,4.430698e+06,31.889000,0.152742,0.498980,1.663000,0.188236
std,4.665692,9.227308e+06,42.367129,0.093451,0.227252,1.872682,0.238201
min,0.000000,4.348350e+04,1.000000,-0.094313,0.006024,0.000000,0.000017
25%,6.000000,8.280438e+05,9.000000,0.079404,0.326045,0.000000,0.022913
50%,9.000000,1.843941e+06,19.000000,0.148398,0.492512,1.000000,0.079621
75%,13.000000,4.587247e+06,40.000000,0.223647,0.684500,2.000000,0.259554
max,23.000000,1.242819e+08,606.000000,0.406067,0.985935,15.000000,0.997778


In [5]:
clientes_pf = gerar_clientes_pf(1000)
clientes_pf.describe()

,idade,renda,pd,comprometimento_renda,historico_atraso,tempo_empresa,patrimonio,tempo_relacionamento
count,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000
mean,39.565000,4573.449198,0.250216,0.291078,2.087000,3.761000,63714.505879,5.123000
std,9.498631,1658.013155,0.285534,0.165076,2.201426,4.221338,39531.703975,3.773607
min,18.000000,1520.396527,0.000074,0.010898,0.000000,0.000000,0.000000,0.000000
25%,33.000000,3386.562682,0.026146,0.161474,1.000000,1.000000,35241.832673,2.000000
50%,40.000000,4353.620020,0.117700,0.267447,2.000000,2.500000,57440.504072,4.000000
75%,46.000000,5476.008122,0.399722,0.396689,3.000000,5.000000,86166.786989,7.000000
max,73.000000,13056.744917,0.998317,0.884559,13.000000,25.000000,273336.004331,24.000000


In [6]:
teste = simular_mes(clientes_pf, clientes_pj, 0.05, 0.03)

In [7]:
print(teste)

{'inad_pf': np.float64(0.049000000000000044), 'inad_pj': np.float64(0.027000000000000024), 'total_inad': np.float64(0.038000000000000034)}


In [8]:
# visualizar_distribuicoes(clientes)
# visualizar_correlacoes(clientes)